# Workflow 1: Evaluate a Target Protein

**Objective:** Assess whether a bacterial or protozoan protein is a viable drug target for computational and wet-lab screening.

**Outputs:**
- Five-criterion evaluation (essentiality, structure, assay feasibility, purification, literature novelty)
- GO/NO-GO recommendation
- Suggested alternatives (if NO-GO)

**Runtime:** ~60 seconds

In [ ]:
import os
import sys
from getpass import getpass

# Supported providers with their configuration requirements
PROVIDERS = {
    "anthropic": {
        "env_key": "ANTHROPIC_API_KEY",
        "model_default": "claude-3-5-sonnet-20241022",
        "requires_base_url": False,
        "description": "Anthropic Direct API (recommended for UT Austin)"
    },
    "openai": {
        "env_key": "OPENAI_API_KEY",
        "model_default": "gpt-4-turbo",
        "requires_base_url": False,
        "description": "OpenAI (including Azure OpenAI via base_url)"
    },
    "cohere": {
        "env_key": "COHERE_API_KEY",
        "model_default": "command-r-plus",
        "requires_base_url": False,
        "description": "Cohere API"
    },
    "bedrock": {
        "env_key": "AWS_ACCESS_KEY_ID",
        "model_default": "claude-3-5-sonnet-20241022",
        "requires_base_url": False,
        "description": "AWS Bedrock (institutional account required)"
    },
    "ollama": {
        "env_key": None,
        "model_default": "llama2",
        "requires_base_url": True,
        "description": "Ollama (local LLMs, no API key needed)"
    },
    "together": {
        "env_key": "TOGETHER_API_KEY",
        "model_default": "meta-llama/Llama-2-7b-chat-hf",
        "requires_base_url": False,
        "description": "Together.ai"
    }
}

print("=" * 70)
print("DRUG DISCOVERY PIPELINE: LLM PROVIDER SETUP")
print("=" * 70)
print()

# Step 1: Select provider
print("1. SELECT YOUR LLM PROVIDER")
print("-" * 70)
for i, (provider_name, info) in enumerate(PROVIDERS.items(), 1):
    print(f"{i}. {provider_name:12} - {info['description']}")

while True:
    try:
        choice = int(input(f"\nEnter choice (1-{len(PROVIDERS)}): "))
        if 1 <= choice <= len(PROVIDERS):
            provider = list(PROVIDERS.keys())[choice - 1]
            break
        else:
            print(f"Invalid choice. Please enter 1-{len(PROVIDERS)}")
    except ValueError:
        print("Invalid input. Please enter a number.")

print(f"\n✓ Selected provider: {provider}")

# Step 2: Get API key (if required)
print("\n2. AUTHENTICATE")
print("-" * 70)
provider_info = PROVIDERS[provider]
api_key = None

if provider_info["env_key"]:
    # Check if key is already in environment
    api_key = os.getenv(provider_info["env_key"])
    if api_key:
        print(f"✓ Found {provider_info['env_key']} in environment (using existing key)")
    else:
        api_key = getpass(f"Enter your {provider} API key (input hidden): ")
        if api_key:
            os.environ[provider_info["env_key"]] = api_key
            print(f"✓ API key set")
        else:
            print(f"⚠ No API key provided. Some providers may not work.")
else:
    print(f"✓ {provider} does not require an API key (local provider)")

# Step 3: Model selection
print("\n3. SELECT MODEL")
print("-" * 70)
model_default = provider_info["model_default"]
model_input = input(f"Enter model name (default: {model_default}): ").strip()
model = model_input if model_input else model_default
print(f"✓ Model: {model}")

# Step 4: Optional base_url (for custom endpoints or Ollama)
base_url = None
print("\n4. OPTIONAL: CUSTOM ENDPOINT (for Ollama, Azure, or custom API)")
print("-" * 70)
if provider == "ollama":
    base_url_default = "http://localhost:11434/v1"
    base_url_input = input(f"Enter Ollama endpoint (default: {base_url_default}): ").strip()
    base_url = base_url_input if base_url_input else base_url_default
    print(f"✓ Endpoint: {base_url}")
else:
    base_url_input = input("Enter custom base_url if needed (leave blank for default): ").strip()
    if base_url_input:
        base_url = base_url_input
        print(f"✓ Custom endpoint: {base_url}")
    else:
        print("✓ Using default endpoint for provider")

# Step 5: Initialize and test client
print("\n5. TESTING CONNECTION")
print("-" * 70)

sys.path.insert(0, '..')
from src import DrugDiscoveryClient, APIConfig

try:
    config = APIConfig(
        provider=provider,
        api_key=api_key,
        model=model,
        base_url=base_url,
    )
    client = DrugDiscoveryClient(config=config)
    print(f"✓ Client initialized successfully")
    print(f"  Provider: {client.config.provider}")
    print(f"  Model: {client.model_id}")
    if base_url:
        print(f"  Endpoint: {base_url}")
except Exception as e:
    print(f"✗ Client initialization failed: {str(e)}")
    print("\nTroubleshooting:")
    if provider == "anthropic":
        print("  - Verify ANTHROPIC_API_KEY is correct (starts with 'sk-ant-')")
    elif provider == "openai":
        print("  - Verify OPENAI_API_KEY is correct (starts with 'sk-')")
    elif provider == "cohere":
        print("  - Verify COHERE_API_KEY is correct")
    elif provider == "ollama":
        print("  - Ensure Ollama is running: ollama serve")
        print("  - Check endpoint is accessible at the specified URL")
    raise

print("\n" + "=" * 70)
print("SETUP COMPLETE - Ready to run workflows!")
print("=" * 70)

# LLM Provider Setup

**Select your LLM provider** and enter credentials below. This setup only needs to run once per session.

Supported providers:
- **anthropic**: Anthropic Direct API (recommended)
- **openai**: OpenAI, Azure OpenAI, or OpenAI-compatible
- **cohere**: Cohere API
- **bedrock**: AWS Bedrock (if institutional account available)
- **ollama**: Local LLMs (Ollama)
- **together**: Together.ai
- **custom**: Any OpenAI-compatible API with custom endpoint

## Step 1: Import and Configure

In [ ]:
import sys
sys.path.insert(0, '..')

from src import DrugDiscoveryClient
import json
from datetime import datetime

# Initialize client (auto-loads from ANTHROPIC_API_KEY or DISCOVERY_PROVIDER environment variables)
client = DrugDiscoveryClient()
print(f"✓ Client initialized")
print(f"  Provider: {client.config.provider}")
print(f"  Model: {client.model_id}")

## Step 2: Define Your Target

In [ ]:
# === EDIT THESE FOR YOUR TARGET ===

organism = "Staphylococcus aureus"  # Bacterial or protozoan species
protein_name = "DNA gyrase subunit B"  # Common or systematic name
protein_id = "GyrB"  # Optional: UniProt ID or common abbreviation

print(f"Evaluating target: {organism} / {protein_name}")

## Step 3: Run Evaluation

In [ ]:
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Running evaluation...\n")

result = client.evaluate_target(
    organism=organism,
    protein_name=protein_name,
    protein_id=protein_id
)

print(f"[{datetime.now().strftime('%H:%M:%S')}] Evaluation complete.\n")

## Step 4: View Results

In [ ]:
# Print the evaluation report
print(result["response"])

## Step 5: Save Results

In [ ]:
# Save to JSON for record-keeping
output_file = f"../examples/{protein_id or protein_name}_evaluation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(output_file, "w") as f:
    json.dump(result, f, indent=2)

print(f"✓ Results saved to: {output_file}")

## Step 6: Next Steps

**If GO recommendation:**
- Proceed to Workflow 2: Generate Validation Controls
- Use the PDB IDs recommended in this evaluation

**If NO-GO recommendation:**
- Review suggested alternatives in the report
- Re-run evaluation for a different protein

**Questions?**
- See README.md for full documentation
- Check examples/ folder for reference outputs